This script categorizes the Loy et al dataset (Plasma cell-free RNA signatures of inflammatory syndromes in children) to keep only infections that have more than 3 patients and the sepsis patients. Infection information was obtained from Dataset S1 (separate file). It also generates a new tab-separated file with the CPM-normalised counts called: GSE255555_pedInflam_filtered_counts.txt.

The metadata of 148 patients are included in the supplementary file so 222 patients were not included due to the absence of metadata

In [1]:
import pandas as pd

# Load metadata file
df = pd.read_csv('pnas.2403897121.sd01.csv')

# Remove leading/trailing spaces from the 'sample_id' column
df['sample_id'] = df['sample_id'].str.strip()

# Count the number of samples for each pathogen
pathogen_counts = df['pathogen'].value_counts()

# Select pathogens that appear more than 3 times
common_pathogens = pathogen_counts[pathogen_counts > 3].index.tolist()

# Select pathogens that contain 'SARS-CoV-2'
sars_cov2_pathogens = df[df['pathogen'].str.contains('SARS-CoV-2', na=False)]['pathogen'].unique().tolist()

# Select patients whose description contains 'sepsis' (case-insensitive)
sepsis_samples = df[df['description'].str.contains('sepsis', case=False, na=False)]['sample_id'].tolist()

# Combine common pathogens and SARS-CoV-2 pathogens, removing duplicates
selected_pathogens = list(set(common_pathogens + sars_cov2_pathogens))

# Filter the dataframe to keep only relevant samples (by pathogen)
filtered_df = df[df['pathogen'].isin(selected_pathogens)]

# Add sepsis samples (if not already included)
filtered_df = pd.concat([filtered_df, df[df['sample_id'].isin(sepsis_samples)]]).drop_duplicates()

# Load the counts file
counts_df = pd.read_csv('GSE255555_pedInflam_counts.csv')

# Extract the sample IDs from the filtered dataframe
selected_sample_ids = filtered_df['sample_id'].tolist()

# Get the columns present in the counts file
counts_columns = counts_df.columns.tolist()

# Identify unmatched sample IDs
unmatched_samples = [sample_id for sample_id in selected_sample_ids if sample_id not in counts_columns]

# Keep only the matched sample ID columns (plus geneID)
filtered_counts_df = counts_df[['geneID'] + [col for col in counts_columns if col in selected_sample_ids]]

# CPM normalization: counts per million for each sample column (exclude 'geneID')
sample_cols = filtered_counts_df.columns.drop('geneID')
counts_only = filtered_counts_df[sample_cols]

# Calculate column sums
col_sums = counts_only.sum(axis=0)

# Compute CPM
cpm = counts_only.div(col_sums, axis=1) * 1_000_000

# Reattach geneID column
filtered_counts_df_cpm = pd.concat([filtered_counts_df['geneID'], cpm], axis=1)

# Print unmatched sample IDs
if unmatched_samples:
    print("\nThe following sample IDs were not found in the counts file:")
    for sample in unmatched_samples:
        print(sample)
else:
    print("\nAll selected sample IDs were successfully matched.")

# Save the CPM-normalized counts dataframe to a tab-separated file
filtered_counts_df_cpm.to_csv('GSE255555_pedInflam_filtered_counts_CPM.txt', sep='\t', index=False)

print("CPM-normalized counts file saved as 'GSE255555_pedInflam_filtered_counts_CPM.txt'.")

# Keep only the metadata for the selected samples
filtered_metadata_df = df[df['sample_id'].isin(selected_sample_ids)]

# Save the filtered metadata to a CSV file
filtered_metadata_df.to_csv('GSE255555_pedInflam_filtered_metadata.csv', index=False)

print("Filtered metadata file saved as 'GSE255555_pedInflam_filtered_metadata.csv'.")


All selected sample IDs were successfully matched.
CPM-normalized counts file saved as 'GSE255555_pedInflam_filtered_counts_CPM.txt'.
Filtered metadata file saved as 'GSE255555_pedInflam_filtered_metadata.csv'.


In [3]:
import pandas as pd

# Load the filtered counts file (tab-separated)
df = pd.read_csv('GSE255555_pedInflam_filtered_counts_CPM.txt', sep='\t')

# Select columns 2 to 10 (index 1 to 9)
columns_to_sum = df.columns[1:10]

# Calculate sum for each column
column_sums = df[columns_to_sum].sum()

# Print sums per column
for col, total in column_sums.items():
    print(f"{col}: {total}")


cfrna_kd_94: 1000000.0000000001
cfrna_kd_10: 999999.9999999999
cfrna_kd_33: 1000000.0000000001
cfrna_kd_18: 1000000.0
cfrna_kd_9: 1000000.0
cfrna_kd_105: 1000000.0
cfrna_kd_5: 1000000.0
cfrna_kd_16: 1000000.0000000001
cfrna_kd_2: 1000000.0


In [4]:
# Print summary counts of selected samples by pathogen
print("\nNumber of selected samples by pathogen:")
print(filtered_metadata_df['pathogen'].value_counts())

# Count how many selected samples mention 'sepsis' in description (case-insensitive)
sepsis_in_filtered = filtered_metadata_df['description'].str.contains('sepsis', case=False, na=False).sum()
print(f"\nNumber of samples with 'sepsis' in description: {sepsis_in_filtered}")

# Show a few example rows to check categories
print("\nExample rows of filtered metadata:")
print(filtered_metadata_df[['sample_id', 'pathogen', 'description']].head(10))

# Optional: cross-tabulation of pathogen vs sepsis presence
filtered_metadata_df['sepsis_flag'] = filtered_metadata_df['description'].str.contains('sepsis', case=False, na=False)
print("\nCross-tabulation of pathogen vs sepsis presence:")
print(pd.crosstab(filtered_metadata_df['pathogen'], filtered_metadata_df['sepsis_flag']))



Number of selected samples by pathogen:
pathogen
Not Available                                                                     45
SARS-CoV-2                                                                        27
Adenovirus                                                                        22
Streptococcus pyogenes                                                             7
MSSA                                                                               6
EBV                                                                                5
SARS-CoV-2, rhino/enterovirus                                                      1
K. oxytoca                                                                         1
SARS-CoV-2, adenovirus, rhino/enterovirus                                          1
SARS-CoV-2, rhinocerebral mucormycosis                                             1
SARS-CoV-2, RSV                                                                    1
Klebsiella pneu